# Journey Console

A local web app for the whole harness: pick a journey (a domain pack under `gmsjourney/policy/`), see what its policy
graph defines, edit the **journey factors** and the rest of the test plan (`project.yaml`) with a live check against
the graph and a preview of the expected conversation, then run the campaign on a chosen device (`cpu`, `cuda:0`–`cuda:3`)
and watch each of the 12 steps report what it produced.

The server runs in a background thread of this kernel; each campaign runs in its own Python process, so the notebook
stays responsive and a run can be cancelled from the page.

In [ ]:
# CELL 1: setup (same requirements as charge_journey_campaign.ipynb: Python 3.12, knowlytix family, a licence)
import os, sys
if os.path.exists("../gmsjourney/__init__.py") and not os.path.exists("gmsjourney/__init__.py"):
    os.chdir("..")                                 # local Jupyter starts in notebooks/: work from the repository root
sys.path.insert(0, os.path.abspath("."))
if not os.environ.get("KNOWLYTIX_LICENSE_KEY") and os.path.exists(os.path.expanduser("~/.knowlytix/license.key")):
    os.environ["KNOWLYTIX_LICENSE_KEY"] = open(os.path.expanduser("~/.knowlytix/license.key")).read().strip()
os.environ.setdefault("KNOWLYTIX_EULA_ACCEPTED", "1")
from gmsjourney import backend
backend.require()

from gmsjourney.app.runs import devices
for d in devices()["options"]:                    # what the Device picker will offer
    print(f"{d['id']:8s} {'available' if d['available'] else 'not visible'}   {d['label']}")

## Open the console

`app.show()` starts the server (port 8790) and displays it here. On JupyterHub it goes through
jupyter-server-proxy automatically; on a plain remote Jupyter, forward the port (`ssh -L 8790:localhost:8790 host`)
and open <http://localhost:8790/> in a browser tab, which gives the console more room than the notebook frame.

In [ ]:
from gmsjourney import app
app.show(port=8790, height=960)

## Without the page

The same run from code. `device` takes `auto`, `cpu` or `cuda:N`; `on_step` is called as each step starts.

In [ ]:
from gmsjourney.pipeline import CampaignConfig, run_campaign
from gmsjourney.domain import set_current
set_current("charge_review_graph")
R = run_campaign(CampaignConfig(out_dir="output/notebook_run", device="auto", max_cases=12,
                                on_step=lambda k, title, R: print(f"step {k}: {title}")))

In [ ]:
app.stop(8790)        # stop the console server